In [ ]:
# Resolve the repo root so `import tools` and the relative config/ and
# output paths work no matter which folder this notebook is launched from.
import os, sys
_d = os.path.abspath(os.getcwd())
while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, 'config')):
    _d = os.path.dirname(_d)
if _d not in sys.path:
    sys.path.insert(0, _d)
os.chdir(_d)
print('repo root:', _d)

# View Production Output

Load and visualize events from production batch output files
(`sensor/`, `step/`, `hits/`). No simulation needed — everything is read
from the HDF5 files.

**Required:** Only the output directory from `run_batch.py`.

## Setup

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

PRODUCTION_DIR = '../dataset_100_wire'  # directory containing sensor/, step/, hits/
DATASET = 'sim'                          # dataset name used in run_batch.py
FILE_INDEX = 0                           # which file (0000, 0001, ...)
EVENT_INDEX = 0                          # event within the file (0-indexed)

# Visualization
THRESHOLD_ENC = 500                      # threshold in electrons for signal display

print(f'Production dir: {PRODUCTION_DIR}')
print(f'Dataset: {DATASET}, file: {FILE_INDEX:04d}, event: {EVENT_INDEX}')

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import numpy as np
import matplotlib.pyplot as plt

from production.load import (
    get_file_paths, load_config, build_viz_config,
    load_event_sensor, load_event_step, load_event_hits,
)
from tools.visualization import (
    visualize_wire_signals,
    visualize_diffused_charge,
    visualize_track_labels,
    get_top_tracks_by_charge,
)


def iter_planes(viz_config):
    """Yield (vol_idx, plane_idx, label) for every active plane."""
    for v, vol in enumerate(viz_config.volumes):
        for p in range(vol.n_planes):
            yield v, p, viz_config.plane_names[v][p]

In [ ]:
# =============================================================================
# FILE PATHS AND METADATA
# =============================================================================

sensor_path, step_path, hits_path = get_file_paths(PRODUCTION_DIR, DATASET, FILE_INDEX)

has_sensor = os.path.exists(sensor_path)
has_step = os.path.exists(step_path)
has_hits = os.path.exists(hits_path)

print(f'sensor: {"found" if has_sensor else "MISSING"}  ({sensor_path})')
print(f'step:   {"found" if has_step else "MISSING"}')
print(f'hits:   {"found" if has_hits else "MISSING"}')

# Load metadata and build viz config
meta = load_config(sensor_path)
viz_config = build_viz_config(sensor_path)

num_time_steps = meta['num_time_steps']
threshold_adc_file = meta['threshold_adc']
electrons_per_adc = meta['electrons_per_adc']

print(f'\nEvents in file: {meta["n_events"]}')
print(f'Grid: {num_time_steps} time x {meta["time_step_us"]} us/bin')
print(f'File threshold: {threshold_adc_file:.2f} ADC ({threshold_adc_file * electrons_per_adc:.0f} e-)')
print(f'Physics: v={meta["velocity_cm_us"]} cm/us, tau={meta["lifetime_us"]} us, recomb={meta["recombination_model"]}')
print(f'Pipeline: noise={meta["include_noise"]}, electronics={meta["include_electronics"]}, digitize={meta["include_digitize"]}')
print(f'Volumes: {viz_config.n_volumes}, planes per volume: '
      f'{[v.n_planes for v in viz_config.volumes]}')

## Load and Visualize Sensor Signals

In [ ]:
# =============================================================================
# LOAD SENSOR SIGNALS
# =============================================================================

dense_signals, event_attrs, pedestals = load_event_sensor(sensor_path, EVENT_INDEX)

n_volumes = int(event_attrs.get('n_volumes', viz_config.n_volumes))
n_per_vol = [int(event_attrs.get(f'n_vol{v}', 0)) for v in range(n_volumes)]
n_deposits = sum(n_per_vol)

print(f'Event {EVENT_INDEX}: {n_deposits:,} deposits  '
      f'(per-volume: {", ".join(f"vol{v}={n:,}" for v, n in enumerate(n_per_vol))})\n')

# Convert to signed ADC if digitized so display sees meaningful amplitudes
if pedestals is not None:
    for k in list(dense_signals.keys()):
        dense_signals[k] = dense_signals[k].astype(np.int32) - pedestals[k]

for v, p, label in iter_planes(viz_config):
    arr = dense_signals.get((v, p))
    if arr is None:
        continue
    n_pix = int(np.count_nonzero(arr))
    mx = float(np.abs(arr).max()) if arr.size else 0.0
    print(f'  vol{v}/{label}: {n_pix:,} pixels, max|ADC|={mx:.1f}')

In [ ]:
if viz_config.plane_names[0][0] != 'Pixel':
    fig = visualize_wire_signals(
        dense_signals, viz_config,
        threshold_enc=THRESHOLD_ENC, gamma=0.2,
    )
    fig.suptitle(f'{DATASET} file {FILE_INDEX:04d}, event {EVENT_INDEX}',
                 fontsize=14, y=1.02)
    plt.show()
else:
    print('Pixel readout — wire-signal viz skipped. '
          'Use visualize_diffused_charge below or a custom 2D projection.')

## Load Energy Deposits (3D Truth Deposits)

In [ ]:
# =============================================================================
# LOAD ENERGY DEPOSIT DATA (per-volume, pure 3D physics)
# =============================================================================
# Per the new schema, step holds positions + de/dx/theta/phi/t0_us +
# charge/photons only. Group/track machinery lives in hits.

step = None
total_de = total_charge = 0.0
n_deps = 0

if has_step:
    step = load_event_step(step_path, EVENT_INDEX)

    for v, vol in enumerate(step):
        n = vol.get('n_actual', 0)
        if n == 0:
            continue
        n_deps += n
        total_de += float(vol['de'].sum())
        total_charge += float(vol['charge'].sum())
        pos = vol['positions_mm']
        print(f'  vol{v}: {n:,} deposits  '
              f'x=[{pos[:,0].min():.0f}, {pos[:,0].max():.0f}] mm  '
              f'sum dE={vol["de"].sum():.1f} MeV  '
              f'sum charge={vol["charge"].sum():.2e} e-')

    print(f'\nTotals: {n_deps:,} deposits, {total_de:.1f} MeV, {total_charge:.2e} e-')
else:
    print('Step file not found.')

## Load Hits (per-particle sensor decomposition) -> track labels + truth charge

In [ ]:
# =============================================================================
# LOAD HITS -> TRACK LABELS + DIFFUSED CHARGE + per-deposit group/qs
# =============================================================================

track_hits = None
truth_dense = None
group_to_track = deposit_to_group = qs_fractions = None

if has_hits:
    track_hits, truth_dense, group_to_track, deposit_to_group, qs_fractions = \
        load_event_hits(hits_path, EVENT_INDEX, num_time_steps,
                        n_volumes=viz_config.n_volumes)

    for v, p, label in iter_planes(viz_config):
        if (v, p) not in track_hits:
            continue
        nl = int(track_hits[(v, p)]['num_labeled'])
        print(f'  vol{v}/{label}: {nl:,} labeled pixels')

    for v in range(viz_config.n_volumes):
        ng = len(group_to_track[v])
        d2g = deposit_to_group[v]
        n_dep = 0 if d2g is None else len(d2g)
        print(f'  vol{v}: {ng:,} groups, {n_dep:,} per-deposit assignments')
else:
    print('Hits file not found — track labels and diffused charge unavailable.')

## Visualize Diffused Charge (Truth Hits)

In [ ]:
if truth_dense is not None:
    fig = visualize_diffused_charge(truth_dense, viz_config, log_norm=True, threshold=50)
    fig.suptitle(f'{DATASET} event {EVENT_INDEX} — Diffused Charge (from correspondence)',
                 fontsize=14, y=1.02)
    plt.show()
else:
    print('No correspondence data for diffused charge visualization.')

## Visualize Track Labels

In [ ]:
top_tracks = []
if track_hits is not None:
    top_tracks = get_top_tracks_by_charge(track_hits, top_n=20)

    if top_tracks:
        print('Top 10 tracks by charge:')
        for i, (tid, charge) in enumerate(top_tracks[:10]):
            print(f'  {i+1:2d}. Track {tid:4d}: {charge:12,.1f}')

    fig = visualize_track_labels(track_hits, viz_config, top_tracks, max_tracks=15)
    n_unique = sum(len(np.unique(g)) for g in group_to_track if g is not None)
    fig.suptitle(f'{DATASET} event {EVENT_INDEX} — Track Labels '
                 f'({n_unique:,} unique tracks across volumes)',
                 fontsize=14, y=1.02)
    plt.show()
else:
    print('No track labels available.')

## Single Track Visualization

In [ ]:
TRACK_RANK = 1  # Nth most active track (1-indexed)

if track_hits is not None and top_tracks and TRACK_RANK <= len(top_tracks):
    selected_tid, selected_charge = top_tracks[TRACK_RANK - 1]
    print(f'Rank {TRACK_RANK}: Track {selected_tid}, charge={selected_charge:,.1f}')

    single_dense = {}
    total_track_hits = 0

    for v, p, label in iter_planes(viz_config):
        if (v, p) not in track_hits:
            continue
        nw = viz_config.volumes[v].num_wires[p]
        dense = np.zeros((nw, num_time_steps), dtype=np.float32)

        data = track_hits[(v, p)]
        nl = int(data['num_labeled'])
        if nl > 0:
            labeled = np.asarray(data['labeled_hits'][:nl])
            tids = np.asarray(data['labeled_track_ids'][:nl])
            mask = tids == selected_tid

            if mask.any():
                wire = labeled[mask, 0].astype(np.int32)
                time_idx = labeled[mask, 1].astype(np.int32)
                charge = labeled[mask, 2]
                valid = (wire >= 0) & (wire < nw) & (time_idx >= 0) & (time_idx < num_time_steps)
                np.add.at(dense, (wire[valid], time_idx[valid]), charge[valid])
                total_track_hits += int(mask.sum())

        single_dense[(v, p)] = dense

    print(f'Track {selected_tid}: {total_track_hits:,} labeled hits')

    fig = visualize_diffused_charge(single_dense, viz_config, log_norm=True, threshold=50)
    fig.suptitle(
        f'{DATASET} event {EVENT_INDEX} — Rank #{TRACK_RANK} Track {selected_tid} '
        f'({total_track_hits:,} hits)',
        fontsize=14, y=1.02)
    plt.show()
else:
    print('Track labels not available or rank not available.')

## Summary

In [ ]:
print('=' * 60)
print(f' Event Summary: {DATASET} file {FILE_INDEX:04d}, event {EVENT_INDEX}')
print('=' * 60)
print(f'  Deposits:       {n_deposits:,}')
if has_step:
    print(f'  Total dE:       {total_de:.2f} MeV')
    print(f'  Total charge:   {total_charge:.2e} e-')

total_pix = sum(int(np.count_nonzero(arr)) for arr in dense_signals.values())
print(f'  Sensor pixels:  {total_pix:,} (file threshold={threshold_adc_file:.1f} ADC)')

if track_hits is not None:
    total_labeled = sum(int(d['num_labeled']) for d in track_hits.values())
    n_unique_tracks = sum(len(np.unique(g)) for g in group_to_track if g is not None)
    print(f'  Labeled pixels: {total_labeled:,}')
    print(f'  Unique tracks:  {n_unique_tracks:,}')

print(f'\n  Physics: v={meta["velocity_cm_us"]} cm/us, '
      f'tau={meta["lifetime_us"]} us, recomb={meta["recombination_model"]}')
print(f'  Pipeline: noise={meta["include_noise"]}, '
      f'electronics={meta["include_electronics"]}, '
      f'digitize={meta["include_digitize"]}')